<a href="https://colab.research.google.com/github/riazhasan998/Data-Science/blob/master/Bangla%20Emotion%20Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade pip
!pip install -q transformers datasets
!pip install openpyxl
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load dataset
from google.colab import drive
import pandas as pd

# 1️⃣ Mount
drive.mount('/content/drive')

# 2️⃣ Read via local path — adjust to your folder structure
df = pd.read_excel(
    "/content/drive/MyDrive/Dataset.xlsx",
    engine="openpyxl"
)

df = df[['Text', 'Emotion']].dropna()

# Encode emotions
label_encoder = LabelEncoder()
df['Emotion_encoded'] = label_encoder.fit_transform(df['Emotion'])

# Split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['Text'].tolist(), df['Emotion_encoded'].tolist(),
    test_size=0.2, stratify=df['Emotion_encoded'], random_state=42
)

In [ ]:
# Check the structure
df.head()

# Check emotion label distribution
df['Emotion'].value_counts()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# Drop rows with missing Emotion or Text
df_cleaned = df[['Text', 'Emotion']].dropna()

# Encode Emotion labels
label_encoder = LabelEncoder()
df_cleaned['Emotion_encoded'] = label_encoder.fit_transform(df_cleaned['Emotion'])

# Visualize class distribution
plt.figure(figsize=(10, 6))
sns.countplot(data=df_cleaned, x='Emotion', order=df_cleaned['Emotion'].value_counts().index)
plt.title("Distribution of Emotion Labels")
plt.xlabel("Emotion")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Display encoded labels mapping
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
label_mapping


In [ ]:
from transformers import AutoTokenizer

model_name = 'sagorsarker/bangla-bert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)


In [ ]:
import torch
from torch.utils.data import Dataset

class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }

    def __len__(self):
        return len(self.labels)

train_dataset = EmotionDataset(train_encodings, train_labels)
test_dataset = EmotionDataset(test_encodings, test_labels)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# Load model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='macro')
    }

# TrainingArguments
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    logging_steps=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Predict
preds = trainer.predict(test_dataset)
y_true = preds.label_ids
y_pred = preds.predictions.argmax(-1)

# Report
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title("BanglaBERT Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:
bangla_out = trainer.predict(test_dataset)
bangla_metrics = compute_metrics((bangla_out.predictions, bangla_out.label_ids))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch
import seaborn as sns
import matplotlib.pyplot as plt

# 1) Tokenizer & encodings
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_enc  = tokenizer(test_texts,  truncation=True, padding=True, max_length=128)

# 2) Datasets
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.encodings = enc
        self.labels = labels
    def __getitem__(self, idx):
        return {
            "input_ids":      torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels":         torch.tensor(self.labels[idx])
        }
    def __len__(self):
        return len(self.labels)

train_ds = EmotionDataset(train_enc, train_labels)
test_ds  = EmotionDataset(test_enc,  test_labels)

# 3) Model + Trainer
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1":       f1_score(p.label_ids, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./mbert_results",
    do_train=True,
    do_eval=True,
    logging_steps=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./mbert_logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 4) Train
trainer.train()

# 5) Evaluate
pred_out = trainer.predict(test_ds)
y_true = pred_out.label_ids
y_pred = pred_out.predictions.argmax(-1)

print("=== mBERT Classification Report ===")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

# 6) Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cmap="Blues")
plt.title("mBERT Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch
import seaborn as sns
import matplotlib.pyplot as plt

# 1) Tokenizer & encodings
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_enc  = tokenizer(test_texts,  truncation=True, padding=True, max_length=128)

# 2) Datasets
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.encodings = enc
        self.labels = labels
    def __getitem__(self, idx):
        return {
            "input_ids":      torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels":         torch.tensor(self.labels[idx])
        }
    def __len__(self):
        return len(self.labels)

train_ds = EmotionDataset(train_enc, train_labels)
test_ds  = EmotionDataset(test_enc,  test_labels)

# 3) Model + Trainer
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1":       f1_score(p.label_ids, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./xlm_results",
    do_train=True,
    do_eval=True,
    logging_steps=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./xlm_logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 4) Train
trainer.train()

# 5) Evaluate
pred_out = trainer.predict(test_ds)
y_true = pred_out.label_ids
y_pred = pred_out.predictions.argmax(-1)

print("=== XLM-RoBERTa Classification Report ===")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

# 6) Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cmap="Blues")
plt.title("XLM-RoBERTa Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()
